# 03_weak_supervision.ipynb

## Setup

In [1]:
import os
import sys
from pathlib import Path

# Mount colab
if True:
    if not os.path.exists('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')

PROJECT_ROOT: Path = Path("/content/drive/MyDrive/python/projects/nlp_projects/med/biomedical-extraction-pipeline")
os.chdir(PROJECT_ROOT)

# Add the PROJECT_ROOT directory to the Python path to import modules
if str(PROJECT_ROOT) in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"ℹ️ Current Python version: {sys.version}")

Mounted at /content/drive
ℹ️ Current Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [2]:
# Install reqs (NOTE: restart after this)
if True:
    %pip install -r {"requirements/03_weak_supervision.txt"}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 79.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.4/252.4 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.4/108.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.9/353.9 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 4.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: jiter
    Found existing installation: jiter 0.15.0
    Uninstalling jiter-0.15.0:
     

In [10]:
# --- src ---
from src.config import load_configs
from src.utils import (
    dynamic_module_reloader,
    seed_everything,
    )
from src.structured_extraction.pydantic_schema import RelationExtraction  

# --- other ---
import pandas as pd
import torch
from dotenv import load_dotenv
from pprint import pprint
from datasets import DatasetDict

USE_LLAMA: bool = False
MODULES_TO_RELOAD = [
    "src",
    "src.config",
    "src.utils",
    "src.model_utils"
]

if USE_LLAMA:
    from src.structured_extraction.llama_cpp import get_response 
    MODULES_TO_RELOAD.append("src.structured_extraction.llama_cpp")
else: 
    from src.structured_extraction.outlines import get_response
    MODULES_TO_RELOAD.append("src.structured_extraction.outlines")

dynamic_module_reloader(modules=MODULES_TO_RELOAD, verbose=1)
config = load_configs()
load_dotenv()
seed_everything(config.env.seed)
pprint(config.to_dict())

# Constants
DEVICE: str = 'cuda' if torch.cuda.is_available() else 'cpu'
SMOKE_TEST: bool = False
TEMPERATURE: float = 0.0
MAX_RETRIES: int = 1
CHECKPOINT_STEPS: int = 25
INSTRUCTION: str = (
    "You are an expert biomedical annotation engine specializing in clinical relation extraction. "
    "Your task is to analyze a text passage and classify the relationship between the target tagged entities.\n\n"

    "ASSIGN LABEL 1 (CAUSATION / ADVERSE EFFECT) IF:\n"
    "- The text explicitly states that the tagged chemical causes, induces, triggers, or results in the tagged disease pathology.\n"
    "- This includes drug-induced side effects, toxicities, or adverse events (e.g., '<chemical>Suxamethonium</chemical> caused severe <disease>fasciculations</disease>'). Even if dose-dependent, side effects are classified as CAUSATION.\n\n"

    "ASSIGN LABEL 2 (THERAPEUTIC / TREATMENT) IF:\n"
    "- The text states that the tagged chemical treats, prevents, mitigates, cures, or manages the disease.\n"
    "- CRITICAL: If a drug has an 'anti-[disease]' effect (e.g., an antihypertensive effect), it means it reduces or treats that condition. Never classify an 'anti-' effect as Label 1.\n\n"

    "ASSIGN LABEL 3 (ACTIVE ANTAGONISM / REVERSAL) IF:\n"
    "- The tagged chemical is the ACTIVE AGENT that blocks, counteracts, inhibits, or reverses an induced drug effect or the action of another chemical.\n"
    "- Example: If 'Chemical A reverses the effect of Chemical B', Chemical A gets Label 3. (Chemical B does NOT get Label 3; it is evaluated on its own primary relationship to the disease).\n\n"

    "ASSIGN LABEL 0 (NONE / POLYSEMY) IF:\n"
    "1. There is no direct causal, therapeutic, or active antagonistic link between the tagged entities.\n"
    "2. MANDATORY POLYSEMY VETO: The tagged disease word is used in a non-pathological, structural, or mechanical sense rather than a clinical illness. For example, if 'depression' refers to a reduction in physiological activity (e.g., 'depression of pacemaker activity'), or if 'failure' is mechanical, you MUST assign Label 0, regardless of what the chemical did to it.\n\n"

    "CRITICAL RULES:\n"
    "- Focus ONLY on the specific directional relationship between the targeted, tagged chemical and tagged disease entities.\n"
    "- Ask yourself: 'What does [Tagged Chemical] do to [Tagged Disease]?'\n"
    "- RELATION BRIDGE RULE: If the tagged chemical affects the disease by blocking, reversing, or inhibiting another chemical (Label 3), you MUST consider that interaction. Otherwise, ignore the actions of completely unrelated, un-tagged chemicals.\n"
    "- Base your decision strictly on the text. Do not let token-matching override clinical logic or prefix qualifiers (like 'anti-')."
    
    "⚠️ MANDATORY OUTPUT FORMATTING DIRECTIVES:\n"
    "1. Your 'chain_of_thought' field MUST be exactly one sentence long.\n"
    "2. Absolutely NO introductory phrases or filler text (e.g., Do NOT write 'Let's break down...', 'Based on the text...', or 'To determine the relationship...').\n"
)

if SMOKE_TEST:
    pass
if USE_LLAMA:
    import llama_cpp
    config.model.local_tagger_checkpoint_large += "-GGUF"
    config.model.local_tagger_checkpoint_medium += "-GGUF"
    config.model.local_tagger_checkpoint_small += "-GGUF"
    print(f"📦 Llama-cpp-python version: {llama_cpp.__version__}")
    print(f"💡 GPU Support compiled in: {llama_cpp.llama_supports_gpu_offload()}")

print("\n\t--- Notebook settings ---")
print(f"Using device: {DEVICE}")
print(f"SMOKE_TEST mode: {SMOKE_TEST}")

Reloaded: src
Reloaded: src.config
Reloaded: src.utils
Reloaded: src.structured_extraction.outlines
✅ Seed set to 42
{'data': {'data_folder': PosixPath('data/documents'),
          'dataset_name': 'bigbio/bc5cdr',
          'flattened_dataset_name': PosixPath('data/processed/bc5cdr_flattened'),
          'relations_dataset_name': PosixPath('data/processed/bc5cdr_relations'),
          'structured_checkpoint_file': PosixPath('data/checkpoints/structured_checkpoint.parquet'),
          'structured_file': PosixPath('data/processed/structured_data.parquet')},
 'env': {'seed': 42},
 'model': {'client_tagger_checkpoint': 'groq/qwen/qwen3-32b',
           'local_tagger_checkpoint_large': 'Qwen/Qwen2.5-32B-Instruct',
           'local_tagger_checkpoint_medium': 'Qwen/Qwen2.5-14B-Instruct',
           'local_tagger_checkpoint_small': 'Qwen/Qwen2.5-7B-Instruct',
           'ner_label_names': ['O',
                               'B-Chemical',
                               'I-Chemical',
         

In [4]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


## Data

In [5]:
# Load the processed dataset (bigbio/BC5CDR)
relationship_dataset = (
    DatasetDict.load_from_disk(config.data.relations_dataset_name)
    # .filter(lambda example: example['chemical_id'] is not None and example['disease_id'] is not None)
    )

relationship_dataset = relationship_dataset.map(
    lambda example: {
        "id": f"{example['document_id']}_{example['chemical_id']}_{example['disease_id']}"
    }
)

relationship_dataset

DatasetDict({
    train: Dataset({
        features: ['document_id', 'chemical', 'disease', 'chemical_id', 'disease_id', 'masked_text', 'label', 'id'],
        num_rows: 5515
    })
    test: Dataset({
        features: ['document_id', 'chemical', 'disease', 'chemical_id', 'disease_id', 'masked_text', 'label', 'id'],
        num_rows: 5528
    })
    validation: Dataset({
        features: ['document_id', 'chemical', 'disease', 'chemical_id', 'disease_id', 'masked_text', 'label', 'id'],
        num_rows: 5317
    })
})

In [11]:
print(os.path.exists(config.data.structured_checkpoint_file))
print(os.path.exists(config.data.structured_file))

False
False


In [12]:
# Call the function with config values
structured_df = get_response(
    df_to_structure=relationship_dataset["train"].to_pandas().head(10),
    id_col="id",
    text_col="masked_text",
    # model_checkpoint=config.model.local_tagger_checkpoint_small,
    model_checkpoint=config.model.client_tagger_checkpoint,
    structured_file=config.data.structured_file,
    structured_checkpoint_file=config.data.structured_checkpoint_file,
    response_model=RelationExtraction,
    instruction=INSTRUCTION,
    temperature=TEMPERATURE,
    max_retries=MAX_RETRIES,
    checkpoint_steps=CHECKPOINT_STEPS,
    use_local_model=False,
    dataset_split="train",
    n_rows_to_process=4,
)

ℹ️ Final processed/structured file not found. Checking for checkpoint...
Starting full instructor/outlines pipeline...
Total items to process: 4


Generating Structured Data:   0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

💾 Checkpoint saved at data/checkpoints/structured_checkpoint.parquet with 4 rows.
Last item processed: 227508_D003000_D007022
💾 Successfully saved 4 rows to data/processed/structured_data.parquet
✅ Merged structured data with original DataFrame. Resulting DataFrame has 10 rows.


In [13]:
structured_df

,document_id,chemical,disease,chemical_id,disease_id,masked_text,label,id,chain_of_thought,weak_label,extraction_status
0,227508,naloxone,hypertensive,D009270,D006973,<chemical> Naloxone </chemical> reverses the a...,0,227508_D009270_D006973,Naloxone reverses the antihypertensive effect ...,3.0,SUCCESS
1,227508,naloxone,hypotensive,D009270,D007022,<chemical> Naloxone </chemical> reverses the a...,0,227508_D009270_D007022,Naloxone reverses the hypotensive effect induc...,3.0,SUCCESS
2,227508,clonidine,hypertensive,D003000,D006973,Naloxone reverses the antihypertensive effect ...,0,227508_D003000_D006973,The text states <chemical>clonidine</chemical>...,2.0,SUCCESS
3,227508,clonidine,hypotensive,D003000,D007022,Naloxone reverses the antihypertensive effect ...,0,227508_D003000_D007022,NaN,NaN,NaN
4,227508,nalozone,hypertensive,None,D006973,Naloxone reverses the antihypertensive effect ...,0,227508_None_D006973,NaN,NaN,NaN
5,227508,nalozone,hypotensive,None,D007022,Naloxone reverses the antihypertensive effect ...,0,227508_None_D007022,NaN,NaN,NaN
6,227508,alpha-methyldopa,hypertensive,D008750,D006973,Naloxone reverses the antihypertensive effect ...,0,227508_D008750_D006973,NaN,NaN,NaN
7,227508,alpha-methyldopa,hypotensive,D008750,D007022,Naloxone reverses the antihypertensive effect ...,1,227508_D008750_D007022,NaN,NaN,NaN
8,354896,lidocaine,cardiac asystole,D008012,D006323,<chemical> Lidocaine </chemical>-induced <dise...,1,354896_D008012_D006323,NaN,NaN,NaN
9,354896,lidocaine,depression,D008012,D003866,<chemical> Lidocaine </chemical>-induced cardi...,0,354896_D008012_D003866,NaN,NaN,NaN


In [13]:
subset_cols = ['document_id', 'chemical', 'disease', 'masked_text', 'chain_of_thought', 'label', 'weak_label']
print(structured_df[subset_cols].to_markdown())

|    |   document_id | chemical         | disease          | masked_text                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

A granular error analysis of our pipeline's weak label generation revealed instances where the LLM-driven four-class schema demonstrated superior clinical fidelity relative to the static human-annotated gold standard. Rather than merely replicating baseline patterns, the teacher engine successfully navigated complex pharmacological phenomena—such as drug antagonism and linguistic polysemy—that frequently confound standard binary classifiers. For example, in documents evaluating central alpha-adrenoceptor stimulation (e.g., PMID: 227508), the model precisely distinguished the active blocking mechanics of antagonists (Label 3) from primary therapeutic trajectories (Label 2). Furthermore, our framework successfully insulated the pipeline against systematic errors introduced by upstream Named Entity Recognition (NER) tools. When confronted with semantic polysemy—such as the mechanical 'depression' of sinoatrial pacemakers mistakenly tagged as a psychiatric disease state—the model invoked our structural veto guardrails to assign a null relation (Label 0), preventing downstream propagation of false clinical correlations. Concurrently, the pipeline identified authentic causal links (e.g., lidocaine-induced bradyarrhythmias) that were omitted by human annotators due to historical specificity biases. These results strongly suggest that leveraging multi-class LLM architectures as weak supervisors provides a robust error-correction layer, uncovering latent clinical insights while filtering out benchmark-induced data artifacts."


In [ ]:
pd.read_parquet("data/checkpoints/structured_checkpoint.parquet")

,chain_of_thought,label,index,extraction_status
0,The text explicitly states that <chemical> Nal...,1,0,SUCCESS
1,The text explicitly states that <chemical> Nal...,1,1,SUCCESS
2,The text explicitly states that naloxone rever...,1,2,SUCCESS
3,The text explicitly states that naloxone rever...,1,3,SUCCESS
